# 03 ? Refer?ncias e compara??o de classificadores
Usamos os conjuntos por utilizador do notebook 02. A compara??o utiliza cinco folds de StratifiedGroupKFold **apenas no treino**, com os mesmos grupos e semente para todos os modelos. A Pipeline completa ? ajustada dentro de cada fold. A valida??o separada permite analisar a transfer?ncia para outros utilizadores.
O F1 de Suspicious ? o crit?rio principal; Average Precision ? secund?rio. Precision e Recall s?o interpretados com as contagens de erros. Quando n?o existem previs?es positivas, a Precision ? indefinida: o c?digo devolve zero por conven??o e assinala essa situa??o.
Os resultados iniciais usam a regra predict de cada algoritmo. A otimiza??o e um eventual ajuste de limiar ser?o feitos exclusivamente no desenvolvimento. O teste final n?o ? avaliado neste notebook.

In [1]:
from pathlib import Path
import sys
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/cybercrime_forensic_dataset.csv').exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src.experiencias import executar_experiencia, tabela_resultados, guardar_resultados
resultados = []

## Refer?ncias simples
A refer?ncia maiorit?ria obt?m accuracy elevada sem detetar qualquer registo suspeito. Prever sempre Suspicious atinge recall total ? custa de sinalizar todos os registos normais. As pontua??es constantes produzem ROC-AUC de 0,5 e Average Precision igual ? preval?ncia positiva.
Estas regras s?o implementadas com DummyClassifier, usando as mesmas entradas e protocolo dos candidatos, para tornar expl?cito o que significa melhorar.

In [2]:
for nome in ["maioritaria", "sempre_suspeita"]:
    resultado = executar_experiencia(nome)
    resultados.append(resultado)
display(tabela_resultados(resultados).round(4))

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
maioritaria,0.000,0.0000,0.000,0.0000,0.0000,0.0,0.8369,0.5,0.1631,0,241
sempre_suspeita,0.285,0.0007,0.285,0.2804,0.1631,1.0,0.1631,0.5,0.1631,1237,0


## Regress?o log?stica
Combina linearmente as caracter?sticas e aplica uma fun??o log?stica para estimar a classe positiva. ? uma refer?ncia explic?vel para rela??es aproximadamente lineares. Usamos C=1, m?ximo de 2 000 itera??es e pesos balanced calculados apenas no treino de cada fold, para estudar o desequil?brio. A escala ? aprendida dentro da Pipeline. A regulariza??o limita coeficientes extremos; intera??es complexas podem n?o ser representadas.

In [3]:
resultado = executar_experiencia("logistica")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferen?a m?dia de F1 CV face ? melhor refer?ncia:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / valida??o:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de valida??o: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
logistica,0.2368,0.0104,0.2946,0.2448,0.1636,0.4855,0.5115,0.5142,0.1671,598,124


Diferen?a m?dia de F1 CV face ? melhor refer?ncia: -0.0482
F1 treino / valida??o: 0.2946 0.2448
Erros de valida??o: FP = 598 ; FN = 124
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## ?rvore de decis?o
Aplica regras sucessivas de separa??o e permite rela??es n?o lineares e intera??es. Esta experi?ncia usa CART, crit?rio Gini, profundidade m?xima 5, pelo menos 20 amostras por folha e pesos balanced. O limite de complexidade reduz a tend?ncia para memorizar dados. ID3, C4.5 e CART s?o fam?lias de ?rvores; a implementa??o usada ? CART para classifica??o, n?o regress?o. Uma ?nica ?rvore pode ter elevada vari?ncia.

In [4]:
resultado = executar_experiencia("arvore")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferen?a m?dia de F1 CV face ? melhor refer?ncia:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / valida??o:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de valida??o: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
arvore,0.2478,0.0424,0.302,0.2819,0.1696,0.834,0.3072,0.5321,0.1743,984,40


Diferen?a m?dia de F1 CV face ? melhor refer?ncia: -0.0373
F1 treino / valida??o: 0.302 0.2819
Erros de valida??o: FP = 984 ; FN = 40
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## Random Forest ? bagging
Combina 100 ?rvores treinadas com amostras bootstrap e subconjuntos de caracter?sticas, reduzindo a vari?ncia de uma ?rvore isolada. Usamos profundidade m?xima 8, m?nimo de 5 registos por folha, pesos balanced e semente 42. Esta abordagem representa ensemble learning por bagging. Permite intera??es, mas n?o cria sinal preditivo que n?o exista; a diferen?a entre treino e valida??o ajuda a discutir sobreajustamento.

In [5]:
resultado = executar_experiencia("random_forest")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferen?a m?dia de F1 CV face ? melhor refer?ncia:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / valida??o:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de valida??o: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
random_forest,0.2283,0.0304,0.4582,0.2169,0.1522,0.3776,0.5555,0.495,0.156,507,150


Diferen?a m?dia de F1 CV face ? melhor refer?ncia: -0.0568
F1 treino / valida??o: 0.4582 0.2169
Erros de valida??o: FP = 507 ; FN = 150
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## k-Nearest Neighbors
Classifica por vota??o dos 15 vizinhos mais pr?ximos, com pesos uniformes. Permite fronteiras locais sem assumir uma rela??o linear. A normaliza??o evita que a escala de um campo domine as dist?ncias. A representa??o one-hot e a dimensionalidade podem tornar as dist?ncias menos informativas; a classe maiorit?ria pode dominar a vota??o. N?o ? alterada a distribui??o do teste.

In [6]:
resultado = executar_experiencia("knn")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferen?a m?dia de F1 CV face ? melhor refer?ncia:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / valida??o:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de valida??o: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
knn,0.0027,0.006,0.0054,0.0,0.0,0.0,0.8363,0.4984,0.1685,1,241


Diferen?a m?dia de F1 CV face ? melhor refer?ncia: -0.2824
F1 treino / valida??o: 0.0054 0.0
Erros de valida??o: FP = 1 ; FN = 241
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## Na?ve Bayes gaussiano
Usa o teorema de Bayes e uma hip?tese simplificadora de independ?ncia condicional das entradas. GaussianNB admite vari?veis cont?nuas, incluindo valores negativos ap?s escala; MultinomialNB n?o ? adequado a esta representa??o. As categorias one-hot e os pares temporais n?o satisfazem necessariamente a hip?tese de independ?ncia. Usamos priors iguais para observar uma op??o sens?vel ao desequil?brio, fixada antes da avalia??o, e o var_smoothing predefinido de 1e-9.

In [7]:
resultado = executar_experiencia("naive_bayes")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferen?a m?dia de F1 CV face ? melhor refer?ncia:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / valida??o:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de valida??o: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
naive_bayes,0.2463,0.01,0.2922,0.2522,0.1646,0.5394,0.4783,0.506,0.1766,660,111


Diferen?a m?dia de F1 CV face ? melhor refer?ncia: -0.0388
F1 treino / valida??o: 0.2922 0.2522
Erros de valida??o: FP = 660 ; FN = 111
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## Support Vector Machine
Procura uma fronteira com margem ampla; o kernel RBF permite separa??o n?o linear. Usamos C=1, gamma=scale e pesos balanced. As entradas s?o escaladas dentro de cada fold. O modelo fornece decision_function para ROC-AUC e Average Precision: estas pontua??es n?o s?o probabilidades calibradas. O custo e a sensibilidade aos par?metros s?o limita??es relevantes.

In [8]:
resultado = executar_experiencia("svm")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferen?a m?dia de F1 CV face ? melhor refer?ncia:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / valida??o:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de valida??o: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
svm,0.2402,0.035,0.5152,0.2101,0.1496,0.3527,0.5677,0.4868,0.1574,483,156


Diferen?a m?dia de F1 CV face ? melhor refer?ncia: -0.0449
F1 treino / valida??o: 0.5152 0.2101
Erros de valida??o: FP = 483 ; FN = 156
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## Gradient Boosting ? boosting
Acrescenta ?rvores pequenas sequencialmente para reduzir a fun??o de perda, ao contr?rio das ?rvores bootstrap do bagging. Usamos 100 etapas, learning_rate=0,05 e profundidade 2. Esta experi?ncia mant?m a distribui??o e os pesos predefinidos para mostrar o comportamento sem pondera??o de classe. Pode aprender intera??es, mas tamb?m ajustar ru?do; exige regulariza??o e valida??o.

In [9]:
resultado = executar_experiencia("gradient_boosting")
resultados.append(resultado)
display(tabela_resultados([resultado]).round(4))
referencia = max(r["cv"]["f1_media"] for r in resultados if r["modelo"] in ["maioritaria", "sempre_suspeita"])
print("Diferen?a m?dia de F1 CV face ? melhor refer?ncia:", round(resultado["cv"]["f1_media"] - referencia, 4))
print("F1 treino / valida??o:", round(resultado["treino"]["f1"], 4), round(resultado["validacao"]["f1"], 4))
print("Erros de valida??o: FP =", resultado["validacao"]["fp"], "; FN =", resultado["validacao"]["fn"])
print("A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
gradient_boosting,0.0,0.0,0.0,0.0,0.0,0.0,0.8363,0.5056,0.1581,1,241


Diferen?a m?dia de F1 CV face ? melhor refer?ncia: -0.285
F1 treino / valida??o: 0.0 0.0
Erros de valida??o: FP = 1 ; FN = 241
A variabilidade dos folds e a utilidade dos alertas devem acompanhar estes valores.


## Registo de resultados de desenvolvimento
O ficheiro pequeno seguinte cont?m m?tricas reais, n?o dados processados nem modelos. As refer?ncias e candidatos s?o avaliados apenas no desenvolvimento. A variabilidade dos folds ? descritiva e n?o constitui um intervalo de confian?a de incidentes independentes.

In [10]:
display(tabela_resultados(resultados).round(4))
guardar_resultados(resultados)
print("Resultados guardados em resultados/desenvolvimento.json; teste final n?o avaliado.")

,F1 CV,Desvio F1 CV,F1 treino,F1 validação,Precision,Recall,Accuracy,ROC-AUC,Average Precision,FP,FN
Modelo,,,,,,,,,,,
maioritaria,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.8369,0.5000,0.1631,0,241
sempre_suspeita,0.2850,0.0007,0.2850,0.2804,0.1631,1.0000,0.1631,0.5000,0.1631,1237,0
logistica,0.2368,0.0104,0.2946,0.2448,0.1636,0.4855,0.5115,0.5142,0.1671,598,124
arvore,0.2478,0.0424,0.3020,0.2819,0.1696,0.8340,0.3072,0.5321,0.1743,984,40
random_forest,0.2283,0.0304,0.4582,0.2169,0.1522,0.3776,0.5555,0.4950,0.1560,507,150
knn,0.0027,0.0060,0.0054,0.0000,0.0000,0.0000,0.8363,0.4984,0.1685,1,241
naive_bayes,0.2463,0.0100,0.2922,0.2522,0.1646,0.5394,0.4783,0.5060,0.1766,660,111
svm,0.2402,0.0350,0.5152,0.2101,0.1496,0.3527,0.5677,0.4868,0.1574,483,156
gradient_boosting,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.8363,0.5056,0.1581,1,241


Resultados guardados em resultados/desenvolvimento.json; teste final n?o avaliado.
